### Import des librairies utiles pour ce projet

In [1649]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### Chargement du dataset selon les csv remis lors de la mission

In [1650]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

### Demandes d'Annabelle
 
Nous souhaitons élaborer différents graphiques autour du chiffre d'affaires comme par exemples l’évolution dans le temps du :

- chiffre d’affaires avec la moyenne mobile (tu pourras choisir la période : jour, semaine, mois, etc.),
- chiffre d’affaires par catégorie,
- nombre de clients par mois,
- nombre de transactions,
- nombre de produits vendus,
- etc.

Il serait également intéressant de faire un zoom sur les références :
- les tops,
- les flops,
- la répartition par catégorie,
- etc.

Enfin, j’aimerais avoir quelques informations sur les profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.

Après, toutes les informations et tous graphiques qui apporteraient de
l’information pertinente sont les bienvenus !

In [1651]:
print('Les informations minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les informations maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

Les informations minimales par colonnes du dataframe sont les suivantes :
id_prod              0_0
date          2021-03-01
session_id           s_1
client_id            c_1
price               0.62
categ                  0
sex                    f
birth               1929
dtype: object

 ----------------------------------------------------------
Les informations maximales par colonnes du dataframe sont les suivantes :
id_prod             2_99
date          2023-02-28
session_id       s_99999
client_id          c_999
price              300.0
categ                  2
sex                    m
birth               2004
dtype: object

 ----------------------------------------------------------
Le type de chaque colonne du dataframe est :
id_prod        object
date           object
session_id     object
client_id      object
price         float64
categ           int64
sex            object
birth           int64
dtype: object


In [1652]:
# Conversion de la colonen date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [1653]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [1654]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [1655]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [1656]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

,date,price,Moyenne glissante (semaine),Moyenne glissante (mois)
0,2021-03-01,16565.22,NaN,NaN
1,2021-03-02,15486.45,NaN,NaN
2,2021-03-03,15198.69,NaN,NaN
3,2021-03-04,15196.07,NaN,NaN
4,2021-03-05,17471.37,NaN,NaN
5,2021-03-06,15785.28,NaN,NaN
6,2021-03-07,14760.20,15780.47,NaN
7,2021-03-08,15679.53,15653.94,NaN
8,2021-03-09,15710.51,15685.95,NaN
9,2021-03-10,15496.87,15728.55,NaN


In [1657]:
figure = px.line(
    df_ca, 
    x='date', 
    y=['Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=px.colors.qualitative.Pastel,
    markers=False,
    title='Évolution de la moyenne glissante du chiffre d\'affaire par semaine et mois',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

#### Chiffre d’affaires par catégorie + évolution dans le temps

In [1658]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [1659]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

Les catégories uniques du df sont les suivantes : [0 1 2]


In [1660]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

Le chiffre d'affaire total de la catégorie 0 est de 4 419 731 euros
Le chiffre d'affaire total de la catégorie 1 est de 4 827 657 euros
Le chiffre d'affaire total de la catégorie 2 est de 2 780 275 euros


In [1661]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ', 
    markers=True,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

#### Nombre de clients par mois + évolution dans le temps

In [1662]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 8600 clients distincts.


In [1663]:
#Affichage du nombre de clients uniques par mois
df_clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
df_clients_par_mois.head()

,date,Nombre de clients
0,2021-03-01,5676
1,2021-04-01,5674
2,2021-05-01,5644
3,2021-06-01,5659
4,2021-07-01,5672


In [1664]:
figure3=px.line(
    df_clients_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [1665]:
#Affichage du nombre de clients par mois
df_clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
df_clients_total_par_mois.head()

,date,Nombre de clients
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [1666]:
figure4=px.line(
    df_clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

#### Nombre de transactions + évolution dans le temps

In [1667]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 687534 transactions.


In [1668]:
#Affichage du nombre de transactions par semaine
df_transactions_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['session_id'].count().reset_index(name='Nombre de transactions par semaine'))
df_transactions_semaine.head()

,date,Nombre de transactions par semaine
0,2021-03-07,6524
1,2021-03-14,6422
2,2021-03-21,6590
3,2021-03-28,6368
4,2021-04-04,6406


In [1669]:
#Affichage du nombre de transactions par mois
df_transactions_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['session_id'].count().reset_index(name='Nombre de transactions par mois'))
df_transactions_mois.head()

,date,Nombre de transactions par mois
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [1670]:
#Affichage du nombre de transactions par trimestre
df_transactions_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['session_id'].count().reset_index(name='Nombre de transactions par trimestre'))
df_transactions_trimestre.head()

### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

,date,Nombre de transactions par trimestre
0,2021-01-01,28601
1,2021-04-01,83578
2,2021-07-01,83702
3,2021-10-01,90790
4,2022-01-01,88633


In [1671]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_transactions= df_transactions_semaine.merge(df_transactions_mois, on = 'date', how='outer').sort_values('date')
df_transactions= df_transactions.merge(df_transactions_trimestre, on = 'date', how='outer').sort_values('date')
df_transactions['Nombre de transactions par mois'] = (df_transactions['Nombre de transactions par mois'].interpolate(method='linear'))
df_transactions['Nombre de transactions par trimestre'] = (df_transactions['Nombre de transactions par trimestre'].interpolate(method='linear'))
df_transactions.head()

,date,Nombre de transactions par semaine,Nombre de transactions par mois,Nombre de transactions par trimestre
0,2021-01-01,NaN,NaN,28601.000000
1,2021-03-01,NaN,28601.0,37763.833333
2,2021-03-07,6524.0,28569.4,46926.666667
3,2021-03-14,6422.0,28537.8,56089.500000
4,2021-03-21,6590.0,28506.2,65252.333333


In [1672]:
figure5=px.line(
    df_transactions,
    x='date',
    y=[ 'Nombre de transactions par semaine','Nombre de transactions par mois','Nombre de transactions par trimestre'],
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure5.show()

#### Nombre de produits vendus + évolution dans le temps

In [1673]:
# tous les produits sur toute la période, puis par mois, puis par semaine
#Affichage du nombre de produits vendus par semaine
df_lapage_prod_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par semaine'))
df_lapage_prod_semaine.head()

,date,Nombre de produits vendus par semaine
0,2021-03-07,1608
1,2021-03-14,1614
2,2021-03-21,1638
3,2021-03-28,1624
4,2021-04-04,1623


In [1674]:
#Affichage du nombre de produits vendus par mois
df_lapage_prod_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par mois'))

In [1675]:
#Affichage du nombre de produits vendus par trimestre
df_lapage_prod_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par trimestre'))

In [1676]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_lapage_prod= df_lapage_prod_semaine.merge(df_lapage_prod_mois, on = 'date', how='outer').sort_values('date')
df_lapage_prod= df_lapage_prod.merge(df_lapage_prod_trimestre, on = 'date', how='outer').sort_values('date')
df_lapage_prod['Nombre de produits vendus par mois'] = (df_lapage_prod['Nombre de produits vendus par mois'].interpolate(method='linear'))
df_lapage_prod['Nombre de produits vendus par trimestre'] = (df_lapage_prod['Nombre de produits vendus par trimestre'].interpolate(method='linear'))
df_lapage_prod.head()

,date,Nombre de produits vendus par semaine,Nombre de produits vendus par mois,Nombre de produits vendus par trimestre
0,2021-01-01,NaN,NaN,2482.000000
1,2021-03-01,NaN,2482.0,2563.666667
2,2021-03-07,1608.0,2484.0,2645.333333
3,2021-03-14,1614.0,2486.0,2727.000000
4,2021-03-21,1638.0,2488.0,2808.666667


In [1677]:
figure6=px.line(
    df_lapage_prod,
    x='date',
    y=[ 'Nombre de produits vendus par semaine','Nombre de produits vendus par mois','Nombre de produits vendus par trimestre'],
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

# Top, Flop, la répartition par catégorie

- les tops,
- les flops,
- la répartition par catégorie

In [1678]:
# Création d'une colonne dans le df principal, indiquant la quantité de produits vendus, par article
df_lapage['quantity'] = df_lapage.groupby('id_prod')['id_prod'].transform('count')
df_lapage.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106


In [1679]:
# Création d'un dataframe restreint contenant la colonne quantité
df_quantity = df_lapage[['id_prod', 'price', 'categ', 'quantity']]
df_quantity.head()

,id_prod,price,categ,quantity
0,0_1259,11.99,0,341
1,0_1390,19.37,0,880
2,0_1352,4.50,0,1004
3,0_1458,6.55,0,1065
4,0_1358,16.49,0,1106


In [1680]:
# Eviter le warning concernant les bugs de pandas sur les copies de dataframes
df_quantity = df_quantity.copy()

df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']

In [1693]:
# Les tops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=False).head(10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])



--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus vendus de la catégorie 0 est le suivant :
---------------------------------------------------------------------------------------------

       id_prod  categ  quantity  price  ca_par_produit
667486  0_1441      0      1235  18.99        23452.65
666021  0_1441      0      1235  18.99        23452.65
669747  0_1441      0      1235  18.99        23452.65
163240  0_1441      0      1235  18.99        23452.65
168178  0_1441      0      1235  18.99        23452.65
167771  0_1441      0      1235  18.99        23452.65
167423  0_1441      0      1235  18.99        23452.65
167115  0_1441      0      1235  18.99        23452.65
687249  0_1441      0      1235  18.99        23452.65
166639  0_1441      0      1235  18.99        23452.65

--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus ve

In [ ]:
# Les flops par catégorie
for categ in df_ca['categ'].unique():
    top10=df_ca[df_ca['categ'] == categ].sort_values('CA_total', ascending=True).head(10)
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print(top10[['id_prod', 'categ','quantity', 'price', 'CA_total']])


Le top 10 des produits les plus vendus de la catégorie 1 est le suivant :
     id_prod  categ  quantity  price  CA_total
3222   1_420      1         2   7.12     14.24
3139   1_224      1         4   4.95     19.80
3134   1_470      1         4   5.41     21.64
2813   1_473      1         9   2.99     26.91
3205   1_404      1         3   9.85     29.55
3067   1_135      1         5   7.99     39.95
2709   1_399      1        11   3.99     43.89
2500   1_511      1        15   2.99     44.85
2766   1_424      1        10   4.91     49.10
2745   1_491      1        10   4.99     49.90
Le top 10 des produits les plus vendus de la catégorie 0 est le suivant :
     id_prod  categ  quantity  price  CA_total
3227  0_1653      0         2   0.99      1.98
3234   0_898      0         2   1.27      2.54
3236  0_1840      0         2   1.28      2.56
3179  0_1759      0         3   0.99      2.97
3195   0_643      0         3   0.99      2.97
3189  0_1191      0         3   0.99      2.97
3218  

#### Profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.